# Qwen 2.5-1.5B-Instruct — Bare-Metal CUDA Inference

This notebook demonstrates single-GPU W4A16 (Group-128) inference for **Qwen 2.5-1.5B-Instruct** using `qwen_engine` on NVIDIA Ampere (RTX 3050 Laptop GPU, SM 8.6).

In [7]:
import sys
import time
from pathlib import Path

src_py = str(Path("src/python").resolve())
if src_py not in sys.path:
    sys.path.insert(0, src_py)

import qwen_engine

print(f"qwen_engine version: {qwen_engine.__version__}")
engine = qwen_engine.Engine("models/qwen25_1.5b_w4a16.qwenbin", "models/tokenizer.bin", device_id=0)

stats = engine.get_memory_stats()
print(f"Weights:   {stats.weights_bytes / (1024**2):.2f} MB")
print(f"KV Cache:  {stats.kv_cache_bytes / (1024**2):.2f} MB")
print(f"Workspace: {stats.workspace_bytes / (1024**2):.2f} MB")
print(f"Total:     {stats.total_allocated_bytes / (1024**2):.2f} MB")

qwen_engine version: 0.2.0
Weights:   759.31 MB
KV Cache:  112.00 MB
Workspace: 8.04 MB
Total:     879.35 MB


## 1. Streaming Chat Generation

In [8]:
engine.reset_session()
messages = [
    {"role": "system", "content": "You are a helpful, concise AI assistant."},
    {"role": "user", "content": "Explain why CUDA Graphs speed up autoregressive LLM decoding in 3 bullet points."}
]

t0 = time.perf_counter()
tokens_out = 0
for chunk in engine.stream(messages, max_tokens=256, temperature=0.0):
    print(chunk, end="", flush=True)
    tokens_out += 1
elapsed = time.perf_counter() - t0
print(f"\n\n--- Generated {tokens_out} chunks in {elapsed:.2f}s ---")

1. **Parallel Processing**: CUDA allows for parallel processing of data across multiple GPUs, which can significantly speed up the decoding process by distributing the computational load among multiple processors.

2. **Memory Bandwidth**: The high bandwidth available through CUDA GPUs enables faster access to and manipulation of data, which is crucial for the efficient decoding of long sequences in language models like those used in large-scale language models.

3. **Optimized for Sequence Processing**: CUDA is specifically optimized for sequence processing tasks, making it particularly well-suited for operations like decoding in language models, where the focus is on processing and manipulating large sets of data in a sequential manner.

--- Generated 128 chunks in 0.78s ---


## 2. Quick Prefill & Decode Throughput Check

In [9]:
engine.reset_session()
prompt = "The quick brown fox jumps over the lazy dog. " * 10
prompt_ids = engine.encode(prompt)

t_pre0 = time.perf_counter()
out = engine.generate(prompt, max_tokens=128, temperature=0.0)
t_total = time.perf_counter() - t_pre0

gen_ids = engine.encode(out)
print(f"Prompt tokens: {len(prompt_ids)}, Generated tokens: {len(gen_ids)}, Total time: {t_total:.3f}s")
print(f"End-to-end throughput: {(len(prompt_ids) + len(gen_ids)) / t_total:.1f} tok/s")

Prompt tokens: 101, Generated tokens: 128, Total time: 0.715s
End-to-end throughput: 320.3 tok/s
